# 04. Improved Experiments and Final Comparison

이 노트북은 baseline 실험 이후 추가로 수행한 개선 실험을 정리합니다.

핵심 목표는 단순히 성능 수치를 조금 올리는 것만이 아니라, 다음 질문에 답하는 것입니다.

1. Threshold tuning이 Accuracy, Precision, Recall, F1-score에 어떤 영향을 주는가?
2. XGBoost와 MLP를 결합한 soft voting ensemble이 단일 모델보다 좋은가?
3. 최종 발표/보고서에서 어떤 모델을 대표 모델로 선택하는 것이 적절한가?

기존 baseline 결과는 유지하고, 추가 실험은 `threshold` suffix로 별도 저장합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.run_improved_experiments import main as run_improved_main
from src.make_model_comparison import make_model_comparison
from src.make_ensemble import make_ensemble

TABLES = PROJECT_ROOT / "results" / "tables"
FIGURES = PROJECT_ROOT / "results" / "figures"

## 개선 실험 실행

아래 명령은 script 기준 실행 예시입니다.

```bash
python src/run_improved_experiments.py --quick
python src/make_model_comparison.py
```

노트북에서는 이미 생성된 CSV를 읽어 분석하는 방식을 기본으로 사용합니다. 시간이 충분하다면 터미널에서 `--quick`을 제거하고 다시 실행할 수 있습니다.

In [ ]:
# 저장된 전체 비교표 확인
comparison = pd.read_csv(TABLES / "model_comparison.csv")
comparison["experiment_label"] = comparison["experiment_label"].fillna("baseline").replace("", "baseline")
comparison[[
    "model", "experiment_label", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc",
    "decision_threshold", "threshold_metric"
]]

## Baseline vs Threshold 비교

Threshold tuning은 예측 확률을 class로 바꾸는 기준을 조정하는 실험입니다. 기본값은 0.5이지만, validation split에서 Accuracy 또는 F1-score가 더 좋은 기준을 찾을 수 있습니다.

단, threshold를 test set에 맞추면 데이터 누수가 되므로 test set은 최종 평가에만 사용해야 합니다.

In [ ]:
threshold_view = comparison[comparison["model"].isin(["XGBoost", "MLP"])].copy()
threshold_view = threshold_view.sort_values(["time_label", "model", "experiment_label"])
threshold_view[[
    "model", "experiment_label", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc",
    "decision_threshold"
]]

In [ ]:
# 15분 데이터 중심 성능 비교
view15 = comparison[comparison["time_label"] == "15minute"].copy()
view15 = view15.sort_values("roc_auc", ascending=False)
view15[["model", "experiment_label", "accuracy", "f1", "roc_auc"]]

## Soft Voting Ensemble

Soft voting ensemble은 XGBoost와 MLP의 예측 확률을 가중 평균하여 최종 예측 확률을 만드는 방식입니다.

본 프로젝트에서는 XGBoost에 0.65, MLP에 0.35의 가중치를 주었습니다. XGBoost가 tabular data에서 더 안정적인 성능을 보였기 때문에 XGBoost 비중을 조금 더 크게 설정했습니다.

Ensemble의 threshold는 test set에 맞추지 않고 0.5로 고정했습니다. 이는 평가 과정에서 데이터 누수를 방지하기 위한 선택입니다.

In [ ]:
ensemble_rows = comparison[comparison["model"] == "SoftVotingEnsemble"]
ensemble_rows[["model", "experiment_label", "time_label", "accuracy", "precision", "recall", "f1", "roc_auc", "decision_threshold"]]

In [ ]:
# Accuracy/F1/ROC-AUC 비교 그래프
plot_df = comparison.copy()
plot_df["experiment_label"] = plot_df["experiment_label"].fillna("baseline").replace("", "baseline")
plot_df["label"] = (
    plot_df["model"].astype(str)
    + "\n"
    + plot_df["time_label"].astype(str)
    + "\n"
    + plot_df["experiment_label"].astype(str)
)
metrics = ["accuracy", "f1", "roc_auc"]

ax = plot_df.set_index("label")[metrics].plot(
    kind="bar",
    figsize=(12, 5),
    ylim=(0.70, 0.92),
    rot=45,
)
ax.set_title("Baseline, Threshold, and Ensemble Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.tight_layout()
plt.show()


## 최종 해석

실험 결과, 가장 일관적인 패턴은 10분 데이터보다 15분 데이터의 성능이 높다는 점입니다. 이는 경기 시간이 진행될수록 골드, 레벨, 오브젝트 차이가 누적되어 최종 승패 신호가 더 명확해지기 때문입니다.

단일 모델 기준으로는 XGBoost 15분 baseline이 가장 안정적인 대표 모델입니다. Accuracy와 F1-score가 높고, feature importance를 통해 어떤 지표가 승패 예측에 중요한지 설명할 수 있습니다.

추가 실험에서는 Soft Voting Ensemble 15분 모델이 가장 높은 ROC-AUC를 기록했습니다. 이는 XGBoost와 MLP가 서로 다른 방식으로 데이터를 학습하며, 두 예측 확률을 결합했을 때 구분 성능이 소폭 개선될 수 있음을 보여줍니다.

따라서 최종 보고서에서는 다음처럼 정리하는 것이 적절합니다.

- **대표 모델:** XGBoost 15분 baseline
- **추가 개선 실험:** Threshold tuning, Soft Voting Ensemble
- **핵심 결론:** 15분 데이터가 10분 데이터보다 승패 예측에 더 유리하며, 골드/레벨/오브젝트 차이가 가장 중요한 예측 신호이다.